# Aula 26: Tratamento de exceções e depuração

## 1. Erros de sintaxe versus exceções

Existem duas categorias de problemas que interrompem a execução de um programa:

- Erros de sintaxe (*syntax errors*): ocorrem quando o código desobedece às regras gramaticais do Python. O interpretador identifica esses erros antes mesmo de iniciar a execução do programa.
- Exceções (*runtime exceptions*): ocorrem quando o código possui sintaxe válida, mas uma operação impossível de ser realizada surge durante a execução (como dividir um número por zero ou tentar converter texto alfabético em `float`).

Sem o tratamento adequado, a ocorrência de uma exceção interrompe abruptamente o programa (*crash*).

In [ ]:
# Exemplo de interrupcao por exceção em tempo de execução (ZeroDivisionError)
amostra = []
media = sum(amostra) / len(amostra)  # gera ZeroDivisionError pois o tamanho é 0

ZeroDivisionError: division by zero

## 2. Tracebacks

Quando uma operação inválida ocorre durante a execução do programa, o Python interrompe o fluxo normal e gera uma exceção. Caso ela não seja tratada, o interpretador exibe um relatório detalhado de erro chamado **traceback**.

Saber ler um traceback é essencial para corrigir problemas:
- As primeiras linhas mostram por onde o fluxo passou (o histórico de chamadas de funções).
- As linhas intermediárias apontam o arquivo e o número exato da linha onde a falha aconteceu.
- A última linha informa a classe da exceção (ex.: `ZeroDivisionError`, `ValueError`, `TypeError`) e uma mensagem explicativa.

In [ ]:
def calcular_media(valores):
    return sum(valores) / len(valores)

amostra_vazia = []
calcular_media(amostra_vazia)  # gera ZeroDivisionError na função calcular_media

ZeroDivisionError: division by zero

## 3. A pilha de chamadas (call stack) e a propagação de exceções (bubbling)

A execução de um programa é organizada em uma **pilha de chamadas** (*call stack*). Sempre que uma função chama outra, a nova função é empilhada no topo; quando ela termina e retorna um valor, é desempilhada.

Quando uma exceção ocorre dentro de uma função aninhada
1. o interpretador verifica se a função atual possui um bloco `try/except` para aquele erro.
2. se não possuir, a função é abortada e a exceção "sobe" para a função que a chamou (processo conhecido como *bubbling* ou propagação).
3. esse processo se repete subindo pela pilha até encontrar um bloco de tratamento.
4. se atingir o escopo principal (nível global) sem ser capturada, o programa encerra com erro fatal.

In [ ]:
def obter_denominador(dados):
    return len(dados) - 1

def soma_dos_quadrados(dados):
    S = 0
    for x in dados:
        S += x ** 2
    return S

def calcular_variancia(dados):
    soma_quadrados = soma_dos_quadrados(dados) - sum(dados) ** 2 / len(dados)
    divisor = obter_denominador(dados)
    return soma_quadrados / divisor

def processar_experimento(dados):
    # A exceção sobe de obter_denominador -> calcular_variancia -> ate ser tratada aqui
    var = calcular_variancia(dados)
    print(f"Variancia calculada: {var:.4f}")

processar_experimento([42.0])  # n = 1 gera divisao por zero no denominador amostral

ZeroDivisionError: float division by zero

## 2. try e except

Para evitar que o programa pare de executar ao encontrar um erro previsto, utilizamos os blocos `try` e `except`.

- `try`: engloba o bloco de código que pode vir a gerar uma exceção.
- `except`: engloba o código que será executado caso a exceção aconteça, permitindo contornar o problema ou registrar uma mensagem adequada.

In [ ]:
dados = []

try:
    media = sum(dados) / len(dados)
    print(f"Media calculada: {media:.4f}")
except:
    print("Aviso: não foi possível calcular a média de uma amostra sem observações.")

Aviso: não foi possível calcular a média de uma amostra sem observações.


## 3. Tratando exceções específicas

Usar uma cláusula `except` genérica (sem especificar o tipo de erro) é uma prática desencorajada, pois pode mascarar falhas inesperadas no código (como erros de digitação de variáveis).

O ideal é capturar explicitamente as exceções que esperamos que possam acontecer:
- `ZeroDivisionError`: divisão por zero.
- `ValueError`: valor com tipo correto, mas conteúdo inadequado (ex.: `float("abc")`).
- `TypeError`: operação entre tipos incompatíveis (ex.: somar número com texto).
- `IndexError`: tentativa de acessar um índice inexistente em uma lista.
- `KeyError`: tentativa de buscar uma chave inexistente em um dicionário.

Também podemos capturar o objeto da exceção utilizando a palavra-chave `as` para inspecionar a mensagem fornecida pelo interpretador.

In [ ]:
def obter_denominador(dados):
    return len(dados) - 1

def soma_dos_quadrados(dados):
    S = 0
    for x in dados:
        S += x ** 2
    return S

def calcular_variancia(dados):
    soma_quadrados = soma_dos_quadrados(dados) - sum(dados) ** 2 / len(dados)
    divisor = obter_denominador(dados)
    return soma_quadrados / divisor

def processar_experimento(dados):
    try:
        var = calcular_variancia(dados)
        print(f"Variancia calculada: {var:.4f}")
    except ZeroDivisionError as err:
        print("Tratamento no nivel do experimento: amostra insuficiente para n - 1.")
        print('Mensagem de erro propagada:', err)

processar_experimento([42.0])  # n = 1 gera divisao por zero no denominador amostral

Tratamento no nivel do experimento: amostra insuficiente para n - 1.
Mensagem de erro propagada: float division by zero


In [ ]:
def converter_medicao(texto):
    try:
        valor = float(texto)
        return valor
    except ValueError as erro:
        print(f"Falha na conversao do registro '{texto}': {erro}")
        return None

In [ ]:
print(converter_medicao("14.85"))
print(converter_medicao("ausente"))

14.85
Falha na conversao do registro 'ausente': could not convert string to float: 'ausente'
None


Podemos também associar múltiplos blocos `except` ao mesmo `try` para tratar causas diferentes de modo distinto:

In [ ]:
def obter_proporcao(lista_dados, indice, divisor):
    try:
        elemento = lista_dados[indice]
        proporcao = elemento / divisor
        return proporcao
    except IndexError:
        print("Erro: indice fornecido esta fora dos limites da amostra.")
        return None
    except ZeroDivisionError:
        print("Erro: denominador nao pode ser zero.")
        return None
    except TypeError:
        print("Erro: os dados devem ser numericos.")
        return None

In [ ]:
valores = [10.0, 25.0, 40.0]
print("Teste 1:", obter_proporcao(valores, 1, 5.0))
print("Teste 2:", obter_proporcao(valores, 5, 2.0))
print("Teste 3:", obter_proporcao(valores, 0, 0.0))

Teste 1: 5.0
Erro: indice fornecido esta fora dos limites da amostra.
Teste 2: None
Erro: denominador nao pode ser zero.
Teste 3: None


Muitas vezes, falhas de tipos diferentes devem receber o mesmo tratamento corretivo. Em vez de duplicar blocos `except` com o mesmo código interno, podemos agrupar as classes de exceção dentro de uma **tupla**:

`except (ExcecaoA, ExcecaoB) as erro:`

- O uso dos parênteses é obrigatório para formar a tupla de exceções.
- O identificador definido após o `as` guarda a instância da exceção que ocorreu no momento, permitindo inspecionar sua mensagem de erro padrão independentemente de ter sido a `ExcecaoA` ou a `ExcecaoB`.

In [ ]:
def converter_para_porcentagem(registro):
    try:
        # Pode falhar com TypeError (se registro for None ou lista)
        # Pode falhar com ValueError (se registro for texto nao numerico)
        valor_decimal = float(registro)
        return f"{valor_decimal * 100.0:.2f}%"
    except (ValueError, TypeError) as erro:
        print(f"Entrada invalida '{registro}'. Detalhe tecnico: {erro}")
        return None

print("Teste valido:", converter_para_porcentagem("0.254"))
print("Teste ValueError:", converter_para_porcentagem("dado_corrompido"))
print("Teste TypeError:", converter_para_porcentagem(None))

Teste valido: 25.40%
Entrada invalida 'dado_corrompido'. Detalhe tecnico: could not convert string to float: 'dado_corrompido'
Teste ValueError: None
Entrada invalida 'None'. Detalhe tecnico: float() argument must be a string or a real number, not 'NoneType'
Teste TypeError: None


## 4. O fluxo: try, except, else e finally

A estrutura de tratamento de exceções pode conter mais duas cláusulas opcionais:

- `else`: executado apenas se o bloco `try` for concluído sem nenhuma exceção.
- `finally`: executado obrigatoriamente, independentemente de ter ocorrido erro ou não (muito comum para fechamento de conexões ou liberação de recursos).

In [ ]:
def analisar_taxa(numerador, denominador):
    try:
        taxa = numerador / denominador
    except ZeroDivisionError:
        print("Falha: divisao por zero detectada.")
    else:
        print(f"Sucesso: taxa calculada em {taxa:.4f}")
    finally:
        print("Encerrando etapa de calculo.")


In [ ]:
analisar_taxa(15.0, 3.0)
print()
analisar_taxa(10.0, 0.0)

Sucesso: taxa calculada em 5.0000
Encerrando etapa de calculo.

Falha: divisao por zero detectada.
Encerrando etapa de calculo.


## 5. Lançando exceções com raise

É recomendado que funções tenham contratos claros: se os dados violam uma premissa matemática, a função poderia mascarar o erro retornando valores arbitrários (como `0` ou `None`), ou então avisar para o chamador que ocorreu algo excepcional.

Regras de domínio comuns:
- Amostras para estimação com divisor $n - 1$ exigem $n \ge 2$.
- Probabilidades devem pertencer ao intervalo $[0, 1]$.
- Pesos em médias ponderadas não podem ser todos nulos ou negativos.

In [ ]:
def calcular_desvio_padrao_amostral(soma_quadrados_desvios, n):
    if n < 2:
        raise ValueError("O tamanho da amostra deve ser no minimo 2 para estimadores amostrais.")
    if soma_quadrados_desvios < 0:
        raise ValueError("A soma dos quadrados dos desvios nao pode ser negativa.")

    variancia = soma_quadrados_desvios / (n - 1)
    return variancia ** 0.5

try:
    dp = calcular_desvio_padrao_amostral(124.5, 1)
    print(f"Desvio padrao: {dp:.4f}")
except ValueError as erro:
    print(f"Erro de validacao estatistica: {erro}")

Erro de validacao estatistica: O tamanho da amostra deve ser no minimo 2 para estimadores amostrais.


## Exercícios

### Exercício 1: Conversão numérica

Em levantamentos empíricos, vetores de dados frequentemente contêm entradas corrompidas por falha de digitação ou preenchimento (ex.: `"12.4"`, `"N/A"`, `None`, `"25.1"`, `"erro"`).

Crie uma função chamada `filtrar_amostra_numerica(dados_brutos)` que
- Itere sobre a lista `dados_brutos`.
- Tente converter cada item individualmente para `float`.
- Capture exceções do tipo `ValueError` e `TypeError`, descartando o elemento inválido sem interromper o laço.
- Retorne uma nova lista contendo apenas os valores numéricos convertidos com sucesso.

In [ ]:
float("N/A") # ValueError: could not convert string to float: 'N/A'

ValueError: could not convert string to float: 'N/A'

In [ ]:
float(None) # TypeError: float() argument must be a string or a real number, not 'NoneType'

TypeError: float() argument must be a string or a real number, not 'NoneType'

In [ ]:
float("erro") # ValueError: could not convert string to float: 'erro'

ValueError: could not convert string to float: 'erro'

In [2]:
# DEFINA filtrar_amostra_numerica AQUI
def filtrar_amostra_numerica(data):
    valid = []
    for e in data:
        try:
            new = float(e)
        except (ValueError, TypeError):
            print(f"Elemento descartado: '{e}'")
        else:
            valid.append(new)
    return valid

registros = ["10.5", "12.0", "ausente", "14.2", None, "erro_medicao", "18.1"]

amostra_limpa = filtrar_amostra_numerica(registros)

print("Amostra limpa:", amostra_limpa)
print(f"Media dos validos: {sum(amostra_limpa) / len(amostra_limpa):.4f}")

Elemento descartado: 'ausente'
Elemento descartado: 'None'
Elemento descartado: 'erro_medicao'
Amostra limpa: [10.5, 12.0, 14.2, 18.1]
Media dos validos: 13.7000


### Exercício 2: Pilha de chamadas

Analise a cadeia de três funções a seguir, que propaga a execução para o cálculo do escore padronizado:

$$\text{Escore relativo} = \frac{x - \min(X)}{\max(X) - \min(X)}$$

Se todos os valores da amostra forem idênticos, a amplitude $\max(X) - \min(X)$ é zero, gerando um `ZeroDivisionError` no final da pilha.

Implemente as três funções de acordo com a hierarquia:
1. `calcular_amplitude(amostra)`: retorna a diferença entre o maior e o menor valor.
2. `escalonar_observacao(x, amostra)`: calcula a amplitude chamando `calcular_amplitude` e retorna $(x - \min(X)) / \text{amplitude}$.
3. `escalonar_vetor(amostra)`: itera sobre cada elemento de `amostra` e chama `escalonar_observacao`. Trate o erro `ZeroDivisionError` neste nível da pilha (onde há contexto suficiente para decidir o que fazer). Caso ocorra a divisão por zero, retorne uma lista de zeros com o mesmo tamanho da amostra original.

Bônus: sugira alterar os parâmetros das funções para evitar cálculos repetidos.

In [5]:
# DEFINA escalonar_vetor AQUI
def calcular_amplitude(amostra):
    return max(amostra) - min(amostra)

def escalonar_observacao(x, amostra):
    return (x - min(amostra)) / (calcular_amplitude(amostra))

def escalonar_vetor(amostra):
    vec = []
    try:
        for x in amostra:
            amp = escalonar_observacao(x, amostra)
            vec.append(amp)
    except ZeroDivisionError:
        return [0] * len(amostra)
    else:
        return vec

amostra_normal = [10.0, 20.0, 30.0, 40.0]
amostra_constante = [5.0, 5.0, 5.0, 5.0]

resultado_1 = escalonar_vetor(amostra_normal)
resultado_2 = escalonar_vetor(amostra_constante)

print("Vetor normal escalonado:", resultado_1)
print("Vetor constante tratado:", resultado_2)

Vetor normal escalonado: [0.0, 0.3333333333333333, 0.6666666666666666, 1.0]
Vetor constante tratado: [0, 0, 0, 0]


### Exercício 3: Raise e try / except / else

Crie uma função para calcular a média e a variância amostral com regras de validação:

`calcular_estatisticas_resumo(dados)`

A função deve validar seus dados de entrada disparando exceções informativas:
- Se `dados` não for uma lista ou tupla, dispare `TypeError("A entrada deve ser uma lista ou tupla.")`.
- Se o tamanho de `dados` for inferior a 2, dispare `ValueError("Amostra deve ter ao menos 2 observacoes para estimativa amostral.")`.
- Se os dados forem válidos, calcule e retorne a tupla `(media, variancia_amostral)`.

No fluxo de testes, processe um lote com diferentes amostras. Trate `(TypeError, ValueError)` capturando a mensagem de erro com `as` e utilize o bloco `else` para formatar e exibir os resultados calculados com sucesso.

In [13]:
# DEFINA calcular_estatisticas_resumo AQUI
def get_media(amostra):
    return sum(amostra) / len(amostra)

def get_var(amostra, avg):
    summ = 0
    for x in amostra:
        summ += (x - avg)**2
    return summ / (len(amostra)-1)

def calcular_estatisticas_resumo(amostra):
    if type(amostra) not in (list, tuple):
        raise TypeError('A entrada deve ser uma lista ou uma tupla.')
    if len(amostra) < 2:
        raise ValueError('Amostra deve ter ao menos 2 observações para estimativa amostral.')
    avg = get_media(amostra)
    return (avg, get_var(amostra, avg))
    

lote_amostras = [
    [10.0, 12.0, 15.0, 14.0, 19.0],
    [42.0],
    "amostra_invalida",
    [100.0, 105.0, 102.0]
]

# ITERE SOBRE O LOTE DE AMOSTRAS AQUI
for amostra in lote_amostras:
    try:
        res = calcular_estatisticas_resumo(amostra)
    except (TypeError, ValueError) as e:
        print(f'\nAmostra <{amostra}> inválida: {e}\n')
    else:
        print(res)

(14.0, 11.5)

Amostra <[42.0]> inválida: Amostra deve ter ao menos 2 observações para estimativa amostral.


Amostra <amostra_invalida> inválida: A entrada deve ser uma lista ou uma tupla.

(102.33333333333333, 6.333333333333333)


### Exercício 4: Distribuição

Para que um conjunto de probabilidades associado a variáveis aleatórias discretas seja válido, duas condições devem ser satisfeitas:
1. Todas as probabilidades individuais devem ser não negativas: $p_i \ge 0$.
2. A soma de todas as probabilidades deve ser igual a 1 (permitindo uma tolerância numérica pequena, como $|1 - \sum p_i| \le 10^{-5}$ devido a arredondamentos de ponto flutuante).

Crie uma função chamada `validar_distribuicao(probabilidades)` que
- Verifique se a lista está vazia; se estiver, dispare um `ValueError` com a mensagem `"A lista de probabilidades nao pode ser vazia."`.
- Verifique se algum valor é negativo; se for, dispare um `ValueError` indicando o valor inválido.
- Verifique se a soma difere de $1$ por mais de $10^{-5}$; se diferir, dispare um `ValueError` informando que a soma é inválida.
- Se todas as regras forem satisfeitas, retorne `None` (nada).

In [18]:
# DEFINA validar_distribuicao AQUI
def validar_distribuicao(probabilidades):
    if len(amostra) == 0:
        raise ValueError('A lista de probabilidades não pode ser vazia')
    summ = 0
    for p in probabilidades:
        if p < 0:
            raise ValueError(f'Valor negativo ({p})')
        summ += p
    if abs(1 - summ) > 1/10**5:
        raise ValueError('Soma inválida')
    return None

distribuicao_valida = [0.20, 0.35, 0.45]
distribuicao_soma_invalida = [0.10, 0.20, 0.50]
distribuicao_negativa = [0.50, -0.10, 0.60]

for dist in [distribuicao_valida, distribuicao_soma_invalida, distribuicao_negativa]:
    try:
        validar_distribuicao(dist)
        print(f"Distribuição {dist} é válida!")
    except ValueError as erro:
        print(f"Erro na distribuição {dist}: {erro}")

Distribuição [0.2, 0.35, 0.45] é válida!
Erro na distribuição [0.1, 0.2, 0.5]: Soma inválida
Erro na distribuição [0.5, -0.1, 0.6]: Valor negativo (-0.1)
